# 가중치 적용 DenseNet121 + U-ignore: Test 평가 및 95% 신뢰구간

- 대상: 클래스 가중치로 학습한 ImageNet 기반 DenseNet121 U-ignore 모델. TorchXRayVision / EVA-X-S 모델용이 아닙니다.
- 기본 체크포인트: runs/uignore_weighted_full/train_20261019_165121_403875/best_model.pth.
- 평가: data/processed/test.csv만 사용. validation / train은 읽지 않습니다.
- 출력: (B, 3, 2), 음성/양성 softmax. 소견 순서는 심비대 / 폐부종 / 흉수입니다.
- 임계값: 사용자 지정 [0.40, 0.40, 0.50] (심비대, 폐부종, 흉수). 기존 0.5 결과와 별도 폴더에 저장합니다.
- 불확실 정답은 해당 소견 평가에서만 제외합니다.
- 환자 단위 cluster bootstrap 1,000회, 95% percentile CI: AUROC, AP, AUPRC, 민감도, 특이도, Precision, F1, Accuracy.
- 그룹별 결과에는 CI를 계산하지 않습니다. 학습을 반복한 변동은 포함하지 않습니다.
- 지표·95% CI·혼동행렬·그룹별 성능·오분류·예측 배열·실행 정보를 저장합니다.
- 서버에서 학습에 사용한 cxr_preprocessing.py와 이미지가 필요합니다. 사전학습 가중치는 다시 다운로드하지 않습니다.


In [ ]:
from pathlib import Path
import json
import sys
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import densenet121, resnet50
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_auc_score,
    average_precision_score, precision_recall_curve, auc,
)
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

FINDINGS = ["Cardiomegaly", "Edema", "Pleural Effusion"]
FINDING_NAMES = ["심비대", "폐부종", "흉수"]
CLASS_NAMES = ["음성", "양성", "불확실"]

## 1. 모델 하나와 평가 데이터 설정

In [ ]:
PROJECT_DIR = Path('/home/team30103/chest-xray-multilabel')
STRATEGY = 'ignore'
ARCHITECTURE = 'densenet121'
IGNORE_HEAD = 'softmax'
MODEL_PATH = PROJECT_DIR / 'runs/uignore_weighted_full/train_20261019_165121_403875/best_model.pth'
IMAGE_ROOT = Path("/home/team30103/physionet_data")
CSV_DIR = PROJECT_DIR / "data" / "processed"
SPLIT_CSVS = {"test": CSV_DIR / "test.csv"}
THRESHOLDS = np.array([0.45, 0.4, 0.5])  # 심비대, 폐부종, 흉수 순서
IMAGE_SIZE = None  # None이면 checkpoint["input"]["height"] 사용; 없으면 직접 지정
BATCH_SIZE = 4
NUM_WORKERS = 0
SAVE_ROOT = (PROJECT_DIR / "evaluation" / "results" / MODEL_PATH.parent.name
             / "densenet121_uignore_weighted_test_95ci_thr045_040_050")

if STRATEGY != "ignore":
    raise ValueError("이 노트북은 U-ignore 전용입니다.")
if ARCHITECTURE not in {"densenet121", "resnet50"}:
    raise ValueError("지원하지 않는 ARCHITECTURE입니다.")
if IGNORE_HEAD not in {"sigmoid", "softmax"}:
    raise ValueError("IGNORE_HEAD는 sigmoid 또는 softmax여야 합니다.")
if THRESHOLDS.shape != (3,) or not np.isfinite(THRESHOLDS).all() or not ((THRESHOLDS >= 0) & (THRESHOLDS <= 1)).all():
    raise ValueError("THRESHOLDS는 0~1 범위의 유한한 값 3개여야 합니다.")
for label, path in [("모델", MODEL_PATH), *SPLIT_CSVS.items()]:
    if not path.is_file():
        raise FileNotFoundError(f"{label} 파일이 없습니다: {path}")
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"이미지 폴더가 없습니다: {IMAGE_ROOT}")
print("모델:", MODEL_PATH)
print("학습 방식:", STRATEGY, "구조:", ARCHITECTURE)

# 환자 단위 cluster bootstrap: 동일 환자의 모든 이미지를 함께 재표집
N_BOOTSTRAP = 1000
CI_LEVEL = 0.95
BOOTSTRAP_SEED = 42
PATIENT_ID_COLUMN = None  # None: subject_id, patient_id 순으로 탐색; 다른 이름이면 직접 지정
if not isinstance(N_BOOTSTRAP, int) or N_BOOTSTRAP < 2:
    raise ValueError("N_BOOTSTRAP은 2 이상의 정수여야 합니다.")
if not 0 < CI_LEVEL < 1:
    raise ValueError("CI_LEVEL은 0과 1 사이여야 합니다.")


## 2. 모델 불러오기

In [ ]:
class CXRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = densenet121(weights=None)
        self.backbone.classifier = nn.Linear(self.backbone.classifier.in_features, 6)

    def forward(self, images):
        return self.backbone(images).reshape(-1, 3, 2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
# 가중치 적용 체크포인트를 기본 U-ignore 체크포인트와 구분합니다.
if checkpoint.get("class_weight_method") is None and checkpoint.get("class_weights") is None:
    raise ValueError("클래스 가중치로 학습된 체크포인트가 아닙니다. MODEL_PATH를 확인하세요.")
if checkpoint.get("smoke_test", False):
    raise ValueError("임시 데이터로 학습한 demo 모델입니다.")
if "target_labels" in checkpoint and list(checkpoint["target_labels"]) != FINDINGS:
    raise ValueError("체크포인트의 소견 순서가 설정과 다릅니다.")
if STRATEGY == "multiclass" and "class_names" in checkpoint:
    if list(checkpoint["class_names"]) != ["negative", "positive", "uncertain"]:
        raise ValueError("클래스 순서가 negative, positive, uncertain인지 확인하세요.")
if STRATEGY == "ignore" and IGNORE_HEAD == "softmax" and "class_names" in checkpoint:
    if list(checkpoint["class_names"]) != ["negative", "positive"]:
        raise ValueError("U-ignore softmax 클래스 순서가 negative, positive인지 확인하세요.")
if IMAGE_SIZE is None:
    IMAGE_SIZE = (checkpoint.get("input") or {}).get("height") or checkpoint.get("config", {}).get("image_size")
if IMAGE_SIZE is None:
    raise ValueError("체크포인트에 입력 크기가 없습니다. IMAGE_SIZE를 학습 크기로 지정하세요.")
model = CXRModel()
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(device).eval()
sys.path.insert(0, str(PROJECT_DIR))
import cxr_preprocessing as preprocessing
# 다른 모델 노트북에서 정규화 전역값을 변경했더라도 학습 설정으로 복원합니다.
input_spec = checkpoint.get('input') or {}
mean = np.asarray(input_spec.get('mean', [0.485, 0.456, 0.406]), dtype=float)
std = np.asarray(input_spec.get('std', [0.229, 0.224, 0.225]), dtype=float)
if mean.shape != (3,) or std.shape != (3,) or not np.isfinite(mean).all() or not np.isfinite(std).all() or (std <= 0).any():
    raise ValueError('정규화 설정이 잘못되었습니다.')
preprocessing.MEAN = tuple(mean.tolist())
preprocessing.STD = tuple(std.tolist())
if list(preprocessing.TARGET_LABELS) != FINDINGS:
    raise ValueError('전처리 소견 순서가 다릅니다.')
CXRDataset = preprocessing.CXRDataset

font_path = PROJECT_DIR / "evaluation" / "MALGUN.TTF"
if font_path.is_file():
    fm.fontManager.addfont(str(font_path))
    plt.rcParams["font.family"] = fm.FontProperties(fname=str(font_path)).get_name()
plt.rcParams["axes.unicode_minus"] = False
print("사용 장치:", device, "입력 크기:", IMAGE_SIZE)

## 3. 예측 수집 및 평가 함수

In [ ]:
def collect_predictions(frame):
    model.eval()
    dataset = CXRDataset(frame, image_size=IMAGE_SIZE)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)
    true_batches, prob_batches = [], []
    with torch.inference_mode():
        for batch_idx, (images, labels) in enumerate(loader, start=1):
            logits = model(images.to(device))
            if STRATEGY == "ignore" and IGNORE_HEAD == "sigmoid":
                probs = logits.sigmoid()
            else:
                probs = logits.softmax(dim=-1)
            true_batches.append(labels.cpu().numpy())
            prob_batches.append(probs.cpu().numpy())
            if batch_idx % 50 == 0 or batch_idx == len(loader):
                print(f"예측 진행: {batch_idx}/{len(loader)} 배치")
    y_true = np.concatenate(true_batches)
    y_prob = np.concatenate(prob_batches)
    if y_true.shape != (len(frame), 3) or not np.isin(y_true, [-100, -1, 0, 1, 2]).all():
        raise ValueError("정답은 (N, 3) 형태이고 -100, -1, 0, 1, 2 코드여야 합니다.")
    y_true = np.where(np.isin(y_true, [-100, -1]), 2, y_true).astype(int)
    expected = (len(frame), 3)
    if STRATEGY == "multiclass":
        expected += (3,)
    elif IGNORE_HEAD == "softmax":
        expected += (2,)
    if y_prob.shape != expected or not np.isfinite(y_prob).all():
        raise ValueError(f"예측 확률 형태 또는 값이 잘못되었습니다: {y_prob.shape}, 기대: {expected}")
    if not ((y_prob >= 0) & (y_prob <= 1)).all():
        raise ValueError("예측 확률이 0~1 범위를 벗어났습니다.")
    if y_prob.ndim == 3 and not np.allclose(y_prob.sum(axis=-1), 1, atol=1e-5):
        raise ValueError("클래스 확률 합이 1이 아닙니다.")
    if STRATEGY == "multiclass":
        denominator = y_prob[:, :, 0] + y_prob[:, :, 1]
        if (denominator <= 0).any():
            raise ValueError("양성·음성 확률 합이 0이어서 정규화할 수 없습니다.")
        score = y_prob[:, :, 1] / denominator
    else:
        score = y_prob if IGNORE_HEAD == "sigmoid" else y_prob[:, :, 1]
    y_pred_binary = (score >= THRESHOLDS[None, :]).astype(int)
    return y_true, y_prob, score, y_pred_binary


def evaluate_predictions(y_true, y_prob, score, y_pred_binary, frame):
    rows, reports, errors, matrices = [], {}, [], []
    for i, finding in enumerate(FINDINGS):
        mask = np.isin(y_true[:, i], [0, 1])
        true, pred, prob = y_true[mask, i], y_pred_binary[mask, i], score[mask, i]
        cm = confusion_matrix(true, pred, labels=[0, 1]) if len(true) else np.zeros((2, 2), dtype=int)
        tn, fp, fn, tp = cm.ravel()
        both_classes = len(np.unique(true)) == 2
        auroc = roc_auc_score(true, prob) if both_classes else np.nan
        ap = average_precision_score(true, prob) if both_classes else np.nan
        auprc = np.nan
        if both_classes:
            precision, recall, _ = precision_recall_curve(true, prob)
            auprc = auc(recall, precision)
        rows.append({
            "finding": finding, "n": int(mask.sum()),
            "excluded_uncertain": int((~mask).sum()),
            "negative_n": int(tn + fp), "positive_n": int(tp + fn),
            "threshold": float(THRESHOLDS[i]), "AUROC": auroc,
            "AP": ap, "AUPRC_trapezoid": auprc,
            "Sensitivity": tp / (tp + fn) if tp + fn else np.nan,
            "Specificity": tn / (tn + fp) if tn + fp else np.nan,
            "Precision": tp / (tp + fp) if tp + fp else np.nan,
            "F1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan,
            "Accuracy": (tp + tn) / len(true) if len(true) else np.nan,
            "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        })
        reports[finding] = classification_report(
            true, pred, labels=[0, 1], target_names=CLASS_NAMES[:2],
            zero_division=0, output_dict=True) if len(true) else {"n": 0, "note": "확실한 정답 없음"}
        matrices.append(cm)
        for idx in np.flatnonzero(mask & (y_true[:, i] != y_pred_binary[:, i])):
            errors.append({"dicom_id": str(frame.iloc[idx]["dicom_id"]),
                           "finding": finding, "true": int(y_true[idx, i]),
                           "pred": int(y_pred_binary[idx, i]),
                           "positive_score": float(score[idx, i])})
    certain = np.isin(y_true, [0, 1]).all(axis=1)
    positive_count = (y_true == 1).sum(axis=1)
    group_rows = []
    for name, subset in {
        "양성 소견 0개": certain & (positive_count == 0),
        "단일 소견 (양성 1개)": certain & (positive_count == 1),
        "복합 소견 (양성 2~3개)": certain & (positive_count >= 2),
    }.items():
        correct = y_true[subset] == y_pred_binary[subset]
        group_rows.append({"group": name, "n": int(subset.sum()),
                           "exact_match": correct.all(axis=1).mean() if subset.any() else np.nan,
                           "finding_accuracy": correct.mean() if subset.any() else np.nan})
    return (pd.DataFrame(rows), reports,
            pd.DataFrame(errors, columns=["dicom_id", "finding", "true", "pred", "positive_score"]),
            pd.DataFrame(group_rows), matrices)

CI_METRICS = ["AUROC", "AP", "AUPRC_trapezoid", "Sensitivity",
              "Specificity", "Precision", "F1", "Accuracy"]


def resolve_patient_column(frame):
    candidates = [PATIENT_ID_COLUMN] if PATIENT_ID_COLUMN else ["subject_id", "patient_id"]
    for column in candidates:
        if column in frame.columns:
            if frame[column].isna().any():
                raise ValueError(f"환자 ID 컬럼 {column}에 결측값이 있습니다.")
            return column
    raise ValueError("환자 단위 CI 계산에 환자 ID가 필요합니다. CSV에 subject_id/patient_id를 포함하거나 PATIENT_ID_COLUMN을 지정하세요.")


def binary_metric_values(true, pred, prob):
    values = dict.fromkeys(CI_METRICS, np.nan)
    if len(true) == 0:
        return values
    tn = np.sum((true == 0) & (pred == 0))
    fp = np.sum((true == 0) & (pred == 1))
    fn = np.sum((true == 1) & (pred == 0))
    tp = np.sum((true == 1) & (pred == 1))
    if len(np.unique(true)) == 2:
        values["AUROC"] = roc_auc_score(true, prob)
        values["AP"] = average_precision_score(true, prob)
        precision, recall, _ = precision_recall_curve(true, prob)
        values["AUPRC_trapezoid"] = auc(recall, precision)
    values["Sensitivity"] = tp / (tp + fn) if tp + fn else np.nan
    values["Specificity"] = tn / (tn + fp) if tn + fp else np.nan
    values["Precision"] = tp / (tp + fp) if tp + fp else np.nan
    values["F1"] = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan
    values["Accuracy"] = (tp + tn) / len(true)
    return values


def bootstrap_confidence_intervals(y_true, score, pred, frame, split):
    column = resolve_patient_column(frame)
    patient_codes, patients = pd.factorize(frame[column].astype(str), sort=True)
    patient_indices = [np.flatnonzero(patient_codes == i) for i in range(len(patients))]
    if len(patients) < 2:
        raise ValueError(f"{split}: 환자 단위 bootstrap에는 최소 2명의 환자가 필요합니다.")
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    samples = {finding: {metric: [] for metric in CI_METRICS} for finding in FINDINGS}
    for repeat in range(N_BOOTSTRAP):
        # 환자를 복원추출하고, 중복 선택된 환자는 이미지도 중복 포함
        selected = rng.integers(0, len(patients), size=len(patients))
        indices = np.concatenate([patient_indices[i] for i in selected])
        for i, finding in enumerate(FINDINGS):
            valid = np.isin(y_true[indices, i], [0, 1])
            idx = indices[valid]
            values = binary_metric_values(y_true[idx, i], pred[idx, i], score[idx, i])
            for metric, value in values.items():
                samples[finding][metric].append(value)
        if (repeat + 1) % 100 == 0 or repeat + 1 == N_BOOTSTRAP:
            print(f"{split} bootstrap: {repeat + 1}/{N_BOOTSTRAP}")
    tail = (1 - CI_LEVEL) / 2
    rows = []
    for i, finding in enumerate(FINDINGS):
        valid = np.isin(y_true[:, i], [0, 1])
        points = binary_metric_values(y_true[valid, i], pred[valid, i], score[valid, i])
        for metric in CI_METRICS:
            values = np.asarray(samples[finding][metric], dtype=float)
            values = values[np.isfinite(values)]
            # 원본 점추정치가 정의되지 않으면 CI도 계산하지 않음
            low, high = (np.quantile(values, [tail, 1 - tail])
                         if len(values) >= 2 and np.isfinite(points[metric]) else (np.nan, np.nan))
            rows.append({"split": split, "finding": finding, "metric": metric,
                         "n": int(valid.sum()), "point_estimate": points[metric],
                         "ci_lower": low, "ci_upper": high, "confidence_level": CI_LEVEL,
                         "valid_bootstrap": len(values), "n_bootstrap": N_BOOTSTRAP,
                         "sampling_unit": "patient", "patient_id_column": column,
                         "patient_count": len(patients)})
            if len(values) < N_BOOTSTRAP * 0.9:
                print(f"주의: {finding} {metric} 유효 반복 {len(values)}/{N_BOOTSTRAP}; CI 해석에 주의하세요.")
    return pd.DataFrame(rows)


## 4. 가중치 모델 Test 평가 및 결과 저장


In [ ]:
# test 데이터의 이미지 경로와 환자 ID를 확인합니다.
frames = {}
for split, csv_path in SPLIT_CSVS.items():
    frame = pd.read_csv(csv_path, dtype={"dicom_id": "string"})
    if frame.empty or "dicom_id" not in frame or frame["dicom_id"].isna().any():
        raise ValueError(f"{split}: CSV가 비었거나 dicom_id가 없거나 결측입니다.")
    if frame["dicom_id"].duplicated().any():
        raise ValueError(f"{split}: 중복 dicom_id가 있습니다.")
    frame["image_path"] = frame["dicom_id"].map(lambda x: str(IMAGE_ROOT / f"{x}.jpg"))
    missing = ~frame["image_path"].map(lambda x: Path(x).is_file())
    if missing.any():
        raise FileNotFoundError(f"{split}: 이미지 {missing.sum()}장 누락. 첫 경로: {frame.loc[missing, 'image_path'].iloc[0]}")
    resolve_patient_column(frame)  # 추론 전에 환자 ID 확인
    frames[split] = frame
all_metrics = []
all_ci = []
for split, frame in frames.items():
    print(f"\n===== {split}: {len(frame)} images =====")
    save_dir = SAVE_ROOT / split
    save_dir.mkdir(parents=True, exist_ok=True)
    y_true, y_prob, score, y_pred_binary = collect_predictions(frame)
    metrics, reports, errors, groups, matrices = evaluate_predictions(
        y_true, y_prob, score, y_pred_binary, frame)
    metrics.insert(0, "split", split)
    all_metrics.append(metrics)
    ci_df = bootstrap_confidence_intervals(y_true, score, y_pred_binary, frame, split)
    all_ci.append(ci_df)
    ci_df.to_csv(save_dir / "confidence_intervals.csv", index=False, encoding="utf-8-sig")
    metrics_with_ci = metrics.copy()
    for metric in CI_METRICS:
        bounds = ci_df[ci_df["metric"] == metric].set_index("finding")
        metrics_with_ci[f"{metric}_ci_lower"] = metrics_with_ci["finding"].map(bounds["ci_lower"])
        metrics_with_ci[f"{metric}_ci_upper"] = metrics_with_ci["finding"].map(bounds["ci_upper"])
    metrics_with_ci.to_csv(save_dir / "metrics_binary_with_ci.csv", index=False, encoding="utf-8-sig")
    display(ci_df)
    display(metrics)
    display(groups)
    for name, table in {"metrics_binary": metrics, "finding_groups": groups,
                        "misclassifications_binary": errors}.items():
        table.to_csv(save_dir / f"{name}.csv", index=False, encoding="utf-8-sig")
    (save_dir / "classification_reports_binary.json").write_text(
        json.dumps(reports, ensure_ascii=False, indent=2), encoding="utf-8")
    np.savez_compressed(save_dir / "predictions.npz", y_true=y_true, y_prob=y_prob,
                        positive_score=score, y_pred_binary=y_pred_binary,
                        dicom_id=frame["dicom_id"].astype(str).to_numpy(dtype=str))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for i, cm in enumerate(matrices):
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", vmin=0, vmax=len(frame),
                    xticklabels=CLASS_NAMES[:2], yticklabels=CLASS_NAMES[:2], ax=axes[i])
        axes[i].set(title=f"{FINDING_NAMES[i]} ({split})", xlabel="예측", ylabel="정답")
    fig.tight_layout()
    fig.savefig(save_dir / "confusion_matrix_binary.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    ax = groups.plot(x="group", y=["exact_match", "finding_accuracy"],
                     kind="bar", rot=0, figsize=(10, 5), ylim=(0, 1.05))
    ax.set_title(f"단일·복합 소견별 성능 ({split})")
    ax.figure.tight_layout()
    ax.figure.savefig(save_dir / "finding_groups.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(ax.figure)

    info = {
        "evaluated_at": datetime.now().astimezone().isoformat(),
        "model_path": str(MODEL_PATH), "architecture": ARCHITECTURE,
        "strategy": STRATEGY, "ignore_head": IGNORE_HEAD if STRATEGY == "ignore" else None,
        "split": split, "evaluation_csv": str(SPLIT_CSVS[split]),
        "image_root": str(IMAGE_ROOT), "n_images": len(frame), "image_size": IMAGE_SIZE,
        "saved_epoch": checkpoint.get("epoch"), "finding_order": FINDINGS,
        "training_config": checkpoint.get("config", {}),
        "training_class_weights": checkpoint.get("class_weights"),
        "model_variant": "DenseNet121 U-ignore weighted",
        "normalization_mean": mean.tolist(), "normalization_std": std.tolist(),
        "class_order": ["negative", "positive"],
        "thresholds": THRESHOLDS.tolist(),
        "uncertain_policy": "소견별 불확실 정답 제외; 복합 소견은 세 소견 모두 확실한 이미지 사용",
        "binary_score": "positive / (negative + positive)" if STRATEGY == "multiclass" else IGNORE_HEAD,
        "binary_rule": "positive_score >= threshold", "bootstrap_ci": True,
        "confidence_intervals": {
            "method": "cluster bootstrap percentile", "confidence_level": CI_LEVEL,
            "n_bootstrap": N_BOOTSTRAP, "seed": BOOTSTRAP_SEED,
            "sampling_unit": "patient", "patient_id_column": resolve_patient_column(frame),
            "patient_count": int(frame[resolve_patient_column(frame)].astype(str).nunique()),
            "metrics": CI_METRICS, "undefined_replicates": "지표별 제외; 유효 반복 횟수 저장",
            "scope": "고정 모델·고정 임계값의 평가 환자 표본 변동; 재학습 변동 미포함",
        },
    }
    (save_dir / "evaluation_info.json").write_text(
        json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    print("저장 완료:", save_dir)

## 5. test 결과 모아 보기

고정된 모델과 임계값의 test 결과를 저장합니다.


In [ ]:
comparison_df = pd.concat(all_metrics, ignore_index=True)
comparison_df.to_csv(SAVE_ROOT / "test_summary.csv", index=False, encoding="utf-8-sig")
display(comparison_df)
ci_comparison_df = pd.concat(all_ci, ignore_index=True)
ci_comparison_df.to_csv(SAVE_ROOT / "test_confidence_intervals.csv",
                         index=False, encoding="utf-8-sig")
display(ci_comparison_df)
